# 2. Train Model - Machine Learning for Smart Greenhouse

This notebook covers data preprocessing, feature engineering, model training, and export for deployment on Edge TPU.

## 1. Load and Explore Sensor Data

Load the sensor_logs.csv file using pandas, explore its structure, check for missing values, and visualize time-series data (temperature, pH, EC, etc.).

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set paths
DATA_PATH = '../../dataset/'
OUTPUT_PATH = '../../models/'

# Create output directory if not exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Load sensor data
sensor_df = pd.read_csv(f'{DATA_PATH}sensor_logs.csv')

# Display basic info
print("Dataset Shape:", sensor_df.shape)
print("\nColumn Names:")
print(sensor_df.columns.tolist())
print("\nFirst 5 rows:")
sensor_df.head()

In [ ]:
# Check for missing values
print("Missing Values:")
print(sensor_df.isnull().sum())
print("\nData Types:")
print(sensor_df.dtypes)
print("\nBasic Statistics:")
sensor_df.describe()

In [ ]:
# Visualize time-series sensor data
fig, axes = plt.subplots(3, 1, figsize=(12, 8))

# Temperature
axes[0].plot(sensor_df['timestamp'], sensor_df['temperature'], color='red', label='Temperature')
axes[0].set_ylabel('Temperature (°C)')
axes[0].legend()
axes[0].set_title('Temperature Over Time')

# Humidity
axes[1].plot(sensor_df['timestamp'], sensor_df['humidity'], color='blue', label='Humidity')
axes[1].set_ylabel('Humidity (%)')
axes[1].legend()
axes[1].set_title('Humidity Over Time')

# CO2
axes[2].plot(sensor_df['timestamp'], sensor_df['co2'], color='green', label='CO2')
axes[2].set_ylabel('CO2 (ppm)')
axes[2].legend()
axes[2].set_title('CO2 Levels Over Time')

plt.tight_layout()
plt.show()

## 2. Load and Preprocess Image Data

Load raw images from raw_images/ folder, resize them, normalize pixel values, and prepare them for model input.

In [ ]:
# Fix typo and continue loading images
if len(image_files) > 0:
    for img_file in image_files[:5]:  # Load first 5 images
        img_path = os.path.join(IMAGE_PATH, img_file)
        img = load_and_preprocess_image(img_path)
        sample_images.append(img)
    
    print(f"Loaded {len(sample_images)} sample images")
    print(f"Image shape: {sample_images[0].shape}")
else:
    print("No images found. Please add images to raw_images/ folder.")

## 3. Merge and Clean Combined Dataset

Combine sensor data with image features, handle missing values, remove outliers, and prepare the final dataset for training.

In [ ]:
# Handle missing values
print("Handling missing values...")
sensor_df_clean = sensor_df.copy()

# Fill numeric columns with median
numeric_cols = sensor_df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if sensor_df_clean[col].isnull().sum() > 0:
        sensor_df_clean[col].fillna(sensor_df_clean[col].median(), inplace=True)

# Remove outliers using IQR method
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

# Apply outlier removal to key columns
for col in ['temperature', 'humidity', 'co2']:
    sensor_df_clean = remove_outliers(sensor_df_clean, col)

print(f"Original rows: {len(sensor_df)}, After cleaning: {len(sensor_df_clean)}")
print("\nCleaned dataset info:")
sensor_df_clean.info()

## 4. Feature Engineering for Time-Series Data

Create derived features such as moving averages, rolling statistics, and time-based features from the sensor time-series data.

In [ ]:
# Feature Engineering for Time-Series Data
sensor_df_fe = sensor_df_clean.copy()

# Convert timestamp to datetime
sensor_df_fe['timestamp'] = pd.to_datetime(sensor_df_fe['timestamp'])

# Extract time-based features
sensor_df_fe['hour'] = sensor_df_fe['timestamp'].dt.hour
sensor_df_fe['day'] = sensor_df_fe['timestamp'].dt.day
sensor_df_fe['month'] = sensor_df_fe['timestamp'].dt.month
sensor_df_fe['day_of_week'] = sensor_df_fe['timestamp'].dt.dayofweek

# Create rolling statistics (moving averages)
window_size = 5
for col in ['temperature', 'humidity', 'co2']:
    sensor_df_fe[f'{col}_ma'] = sensor_df_fe[col].rolling(window=window_size).mean()
    sensor_df_fe[f'{col}_std'] = sensor_df_fe[col].rolling(window=window_size).std()

# Create lag features
for col in ['temperature', 'humidity', 'co2']:
    sensor_df_fe[f'{col}_lag1'] = sensor_df_fe[col].shift(1)
    sensor_df_fe[f'{col}_lag2'] = sensor_df_fe[col].shift(2)

# Drop rows with NaN from rolling/lag operations
sensor_df_fe = sensor_df_fe.dropna()

print("Feature Engineered Dataset:")
print(f"Shape: {sensor_df_fe.shape}")
print("\nNew Features:")
print(sensor_df_fe.columns.tolist())

## 5. Train Random Forest / ANN Model

Split data into training and testing sets, train Random Forest or ANN model using scikit-learn or TensorFlow, and tune hyperparameters.

In [ ]:
# Import ML libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Prepare features and target
# Example: Predict temperature based on other sensor readings
feature_cols = ['humidity', 'co2', 'hour', 'day_of_week', 
                'temperature_ma', 'temperature_std', 'temperature_lag1', 'temperature_lag2']
target_col = 'temperature'

X = sensor_df_fe[feature_cols]
y = sensor_df_fe[target_col]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
# Train Random Forest Model
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_train = rf_model.predict(X_train_scaled)
y_pred_test = rf_model.predict(X_test_scaled)

# Evaluate model
print("=== Random Forest Model Performance ===")
print(f"\nTraining Set:")
print(f"  R² Score: {r2_score(y_train, y_pred_train):.4f}")
print(f"  MAE: {mean_absolute_error(y_train, y_pred_train):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}")

print(f"\nTest Set:")
print(f"  R² Score: {r2_score(y_test, y_pred_test):.4f}")
print(f"  MAE: {mean_absolute_error(y_test, y_pred_test):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")

## 6. Evaluate Model Performance

Evaluate the trained model using metrics like accuracy, precision, recall, F1-score, and confusion matrix. Visualize results.

In [ ]:
# Visualize Model Performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Actual vs Predicted
axes[0].scatter(y_test, y_pred_test, alpha=0.5, color='blue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Temperature')
axes[0].set_ylabel('Predicted Temperature')
axes[0].set_title('Actual vs Predicted Temperature (Test Set)')

# Plot 2: Feature Importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=True)

axes[1].barh(feature_importance['feature'], feature_importance['importance'], color='green')
axes[1].set_xlabel('Importance')
axes[1].set_title('Feature Importance')

plt.tight_layout()
plt.show()

# Print feature importance
print("\nFeature Importance Ranking:")
print(feature_importance.sort_values('importance', ascending=False).to_string(index=False))

## 7. Export Model to TensorFlow Lite

Convert the trained model to TensorFlow Lite format (.tflite) for deployment on Edge TPU and save to models/ folder.

In [ ]:
# Export Model using joblib (for scikit-learn models)
import joblib

# Save the model and scaler
joblib.dump(rf_model, f'{OUTPUT_PATH}greenhouse_rf_model.joblib')
joblib.dump(scaler, f'{OUTPUT_PATH}feature_scaler.joblib')

print(f"Random Forest model saved to: {OUTPUT_PATH}greenhouse_rf_model.joblib")
print(f"Feature scaler saved to: {OUTPUT_PATH}feature_scaler.joblib")

# Optional: Export to TensorFlow Lite format (if using TensorFlow/Keras)
# Uncomment below if you have a Keras model
"""
import tensorflow as tf

# Convert to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
tflite_model = converter.convert()

# Save TFLite model
with open(f'{OUTPUT_PATH}greenhouse_model.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"TFLite model saved to: {OUTPUT_PATH}greenhouse_model.tflite")
"""

print("\n=== Model Export Complete ===")
print(f"Models saved in: {OUTPUT_PATH}")

In [ ]:
# Import image processing libraries
from PIL import Image
import cv2
from sklearn.preprocessing import LabelEncoder

# Set image parameters
IMG_SIZE = (224, 224)  # Standard size for CNN models
IMAGE_PATH = f'{DATA_PATH}raw_images/'

# Function to load and preprocess images
def load_and_preprocess_image(image_path, target_size=IMG_SIZE):
    """Load image, resize, and normalize pixel values."""
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, target_size)
    img = img.astype('float32') / 255.0  # Normalize to [0, 1]
    return img

# Example: Load sample images (if available)
sample_images = []
image_files = [f for f in os.listdir(IMAGE_PATH) if f.endswith(('.png', '.jpg', '.jpeg'))]

print(f"Found {len(image_files)} images in {IMAGE_PATH}")

if len(image_images) > 0:
    for img_file in image_files[:5]:  # Load first 5 images
        img_path = os.path.join(IMAGE_PATH, img_file)
        img = load_and_preprocess_image(img_path)
        sample_images.append(img)
    
    print(f"Loaded {len(sample_images)} sample images")
    print(f"Image shape: {sample_images[0].shape}")
else:
    print("No images found. Please add images to raw_images/ folder.")